# Dallas CIP Parser

Converts all PDFs in `Dallas/PDF/` to CSVs in `Dallas/CSV/`.

**Format families detected from sampling:**
- **Format A** (2007–2016): Portrait, no Unit Number column, FY cols like `FY2007-08`
- **Format A'** (2017): Portrait, same structure, `Future Cost` instead of explicit Total
- **Format B** (2018): Landscape (rotation=270°), explicit `Unit Number` + `Capital Adopted` cols
- **Format C** (2019–2025): Portrait, project ID embedded in name as ` - XXXX` suffix

**Column mapping (per user spec):**
- `Project` / `Project Name` → `project_name`
- `Unit Number` → `project_id` (Format B explicit; Format C extracted from name suffix)
- `Capital Adopted` / `Budget as of date` / `Budget ITD` → `previous_appropriations`
- `FY YYYY-YY` → `year_{second_year}` (e.g. `FY2018-19` → `year_2019`)
- `Service` → `project_type`
- `Completion Date` / `In Service Date` → `end_year`
- `Total [Estimated] Cost` / `Total Budget` / `Total Project Cost` → `project_total`

In [ ]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path

PDF_DIR = Path('Dallas/PDF')
CSV_DIR = Path('Dallas/CSV')
CSV_DIR.mkdir(exist_ok=True)

BASE_COLS = [
    'cip_year', 'project_type', 'source_page', 'department',
    'project_name', 'project_id', 'address_location',
    'start_year', 'end_year', 'project_description',
    'project_justification', 'previous_appropriations', 'project_total'
]

print('PDFs found:', sorted(p.name for p in PDF_DIR.glob('*.pdf')))

## Cell 2 — Format detection & sampling

Print the first confirmed project page from each PDF to verify format routing.

In [ ]:
def detect_format(stem: str) -> str:
    """Route by year."""
    y = int(stem)
    if y <= 2016: return 'A'
    if y == 2017: return 'Aprime'
    if y == 2018: return 'B'
    return 'C'

def is_project_page_a(text: str) -> bool:
    """Format A/A': must have at least one FY column keyword and a budget figure."""
    return bool(re.search(r'FY\s*\d{4}', text)) and bool(re.search(r'\d{3,}', text))

def is_project_page_b(pg) -> bool:
    """Format B: rotated landscape pages only."""
    return pg.rotation == 270

def is_project_page_c(text: str) -> bool:
    """Format C: must have FY column and Budget column labels."""
    return bool(re.search(r'FY\s*\d{4}', text)) and bool(
        re.search(r'Budget|ITD|Appropriat', text, re.I))

# Sampling: show first REAL project page from each PDF using find_hdr
# (Note: find_hdr is defined in the helpers cell — run that cell first)
for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    stem = pdf_path.stem
    fmt  = detect_format(stem)
    with pdfplumber.open(pdf_path) as pdf:
        for i, pg in enumerate(pdf.pages):
            text = pg.extract_text() or ''
            if fmt == 'B':
                if is_project_page_b(pg):
                    print(f'{stem} (Format {fmt}) — first project page: {i+1}, rotation={pg.rotation}')
                    break
            elif fmt in ('A', 'Aprime'):
                if not is_project_page_a(text): continue
                tbl = pg.extract_table()
                if not tbl: continue
                hdr_idx = find_hdr(tbl)
                if hdr_idx is None: continue
                hdrs = [str(c or '').replace('\n', ' ').strip() for c in tbl[hdr_idx]]
                first_data = next(
                    (str(row[0] or '').replace('\n',' ').strip()
                     for row in tbl[hdr_idx+1:]
                     if str(row[0] or '').strip()), '?')
                print(f'{stem} (Format {fmt}) — first project page: {i+1}')
                print(f'  Headers: {hdrs[:5]}')
                print(f'  First row name: {first_data[:60]}')
                break
            else:  # C
                if not is_project_page_c(text): continue
                tbl = pg.extract_table()
                if not tbl: continue
                hdr_idx = find_hdr(tbl)
                if hdr_idx is None: continue
                hdrs = [str(c or '').replace('\n', ' ').strip() for c in tbl[hdr_idx]]
                first_data = next(
                    (str(row[0] or '').replace('\n',' ').strip()
                     for row in tbl[hdr_idx+1:]
                     if str(row[0] or '').strip()
                     and not re.match(r'Total|Subtotal|Fund', str(row[0] or ''), re.I)), '?')
                print(f'{stem} (Format {fmt}) — first project page: {i+1}')
                print(f'  Headers: {hdrs[:5]}')
                print(f'  First row name: {first_data[:60]}')
                break


## Cell 3 — Helper functions

In [ ]:
def parse_fy_header(h: str):
    """'FY2007-08', 'FY 2018-19', 'FY 2022-23 Budget' → 'year_2008', 'year_2019', etc.
    Second year of the fiscal range is used (per Dallas convention)."""
    h = (h or '').replace('\n', ' ')
    m = re.search(r'FY\s*(\d{4})[-–](\d{2,4})', h, re.I)
    if not m:
        return None
    y1, y2 = m.group(1), m.group(2)
    full_y2 = (y1[:2] + y2) if len(y2) == 2 else y2
    return f'year_{full_y2}'

def clean_num(s):
    """Strip currency symbols/commas; return '' for non-numeric values."""
    if not s:
        return ''
    s = str(s).strip()
    if re.fullmatch(r'[\-–—$%TBDNAvarious Ongoing]*', s, re.I):
        return ''
    s = re.sub(r'[\$,\s]', '', s)
    s = s.replace('(', '-').replace(')', '')
    return s if re.fullmatch(r'-?\d+\.?\d*', s) else ''

def extract_end_year(s):
    """'06/2023', '9/1/2022', '3rd/05', '09/08' → year string or ''."""
    if not s or str(s).strip().lower() in ('tbd', 'various', 'ongoing', 'n/a', ''):
        return ''
    s = str(s).strip()
    m4 = re.search(r'(20\d{2}|19\d{2})', s)
    if m4:
        return m4.group(1)
    # '3rd/05', '09/08' — 2-digit year, assume 2000s
    m2 = re.search(r'/(\d{2})\s*$', s)
    if m2:
        return '20' + m2.group(1)
    return ''

def extract_pid_from_name(name: str):
    """'Arapaho Rd - V939 Program' → ('Arapaho Rd', 'V939').
    Matches '- XXXX' suffix where XXXX is 2–5 uppercase alphanumeric."""
    name = (name or '').replace('\n', ' ').strip()
    m = re.search(r'\s+[-–]\s+([A-Z][A-Z0-9]{1,4})\s*(?:Program)?\s*$', name)
    if m:
        return name[:m.start()].strip(), m.group(1)
    return name, None

def get_dept_from_text(text: str, fmt: str) -> str:
    """Extract department from page-level text above the table.
    Format A/A': ALL-CAPS section banner (e.g. 'STREET AND THOROUGHFARE CAPITAL IMPROVEMENTS').
    Format C:   Title-case section heading (e.g. 'City Facilities').
    Format B:   Handled separately via reversed section word."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if fmt in ('A', 'Aprime'):
        for line in lines:
            # Long all-caps line that ends with 'IMPROVEMENTS' or 'PROGRAM'
            if (line.isupper() and len(line) > 10 and
                    re.search(r'IMPROVEMENT|PROGRAM|FACILIT|DRAINAGE|WATER|LIBRARY', line)):
                # Strip trailing 'CAPITAL IMPROVEMENTS' etc.
                dept = re.sub(r'\s+CAPITAL\s+IMPROVEMENTS?\s*$', '', line, flags=re.I).strip()
                return dept.title()
    elif fmt == 'C':
        # First non-empty line that is NOT 'Project List' and NOT a table header keyword
        skip = re.compile(r'^(Project\s*List|Council|Service|Funding|Budget|FY|Total)', re.I)
        for line in lines:
            if not skip.match(line) and len(line) > 3:
                return line
    return ''

def find_hdr(tbl):
    """Locate the header row in a table.
    Requires first cell == 'Project' or 'Projects' (exact, case-insensitive)
    AND an FY YYYY pattern in the row AND Service|Funding|Council|Budget in the row.
    This avoids false positives on summary pages where 'Total Project Cost'
    appears in a non-header row."""
    for i, r in enumerate(tbl):
        c0 = str(r[0] or '').replace('\n', ' ').strip()
        if not re.match(r'^projects?$', c0, re.I):
            continue
        row_str = ' '.join(str(c or '') for c in r)
        if not re.search(r'FY\s*\d{4}', row_str):
            continue
        if re.search(r'Service|Funding|Council|Budget', row_str, re.I):
            return i
    return None

print('Helpers defined.')


## Cell 4 — Format A parser (2007–2016)

In [ ]:
def parse_format_a(pdf_path: Path, cip_year: str) -> pd.DataFrame:
    """Parse Format A (2007-2016) PDFs.
    Table columns: Project | Service | Key Focus Area | Council District |
                   Funding Source | Budget as of [date] | Spent | Remaining |
                   FY YYYY-YY ... | Total Estimated Cost | In Service Date
    """
    records = []
    current_dept = ''
    year_cols_cache = {}   # page → list of year_YYYY col names

    with pdfplumber.open(pdf_path) as pdf:
        for pg_num, pg in enumerate(pdf.pages, start=1):
            text = pg.extract_text() or ''
            if not is_project_page_a(text):
                continue

            # Update department from page banner
            dept = get_dept_from_text(text, 'A')
            if dept:
                current_dept = dept

            tbl = pg.extract_table()
            if not tbl or len(tbl) < 2:
                continue

            # Find header row using strict first-cell check
            hdr_idx = find_hdr(tbl)
            if hdr_idx is None:
                continue

            headers = [str(c or '').replace('\n', ' ').strip() for c in tbl[hdr_idx]]

            # Map column indices
            col_map = {}
            year_cols = []  # list of (col_idx, year_YYYY)
            for i, h in enumerate(headers):
                hl = h.lower()
                if re.search(r'^project\s*(name)?$', hl):
                    col_map['project_name'] = i
                elif re.search(r'service', hl):
                    col_map['project_type'] = i
                elif re.search(r'council', hl):
                    col_map['council_district'] = i
                elif re.search(r'funding', hl):
                    col_map['funding_source'] = i
                elif re.search(r'budget|capital adopted', hl) and 'previous_appropriations' not in col_map:
                    col_map['previous_appropriations'] = i
                elif re.search(r'total.*cost|total.*budget|total.*estimat', hl):
                    col_map['project_total'] = i
                elif re.search(r'in.*service|service.*date|completion|comp.*date', hl):
                    col_map['end_year'] = i
                fy = parse_fy_header(h)
                if fy:
                    year_cols.append((i, fy))

            if 'project_name' not in col_map:
                col_map['project_name'] = 0   # fallback

            # Collect year column names for this page (for final wide-table assembly)
            year_col_names = [y for _, y in year_cols]

            # Parse data rows
            for row in tbl[hdr_idx + 1:]:
                if not any(row):
                    continue
                cells = [str(c or '').replace('\n', ' ').strip() for c in row]
                name = cells[col_map['project_name']] if col_map.get('project_name') is not None else ''
                # Skip rows that look like sub-headers or page footers
                if not name or re.match(r'^(Total|Subtotal|Page|Fund)', name, re.I):
                    continue
                # Must have at least one non-zero numeric in FY columns to be a project row
                if not any(re.search(r'\d', cells[i]) for i, _ in year_cols):
                    budget_col = col_map.get('previous_appropriations')
                    if budget_col is None or not re.search(r'\d', cells[budget_col]):
                        continue

                rec = {
                    'cip_year': cip_year,
                    'source_page': pg_num,
                    'department': current_dept,
                    'project_name': name,
                    'project_id': None,
                    'address_location': None,
                    'start_year': None,
                    'project_description': None,
                    'project_justification': None,
                }
                rec['project_type']            = cells[col_map['project_type']] if 'project_type' in col_map else None
                rec['previous_appropriations'] = clean_num(cells[col_map['previous_appropriations']]) if 'previous_appropriations' in col_map else None
                rec['project_total']           = clean_num(cells[col_map['project_total']]) if 'project_total' in col_map else None
                rec['end_year']                = extract_end_year(cells[col_map['end_year']]) if 'end_year' in col_map else None
                for col_i, yr_name in year_cols:
                    rec[yr_name] = clean_num(cells[col_i])
                records.append(rec)

    df = pd.DataFrame(records)
    return df

print('Format A parser defined.')

## Cell 5 — Format A' parser (2017)

In [ ]:
def parse_format_aprime(pdf_path: Path, cip_year: str) -> pd.DataFrame:
    """Parse Format A' (2017) PDF.
    Columns: Project Name | Service | Key Focus Area | Council District |
             Funding Source | Budget | Spent | Remaining |
             FY2017-18 Adopted | FY2018-19 Planned | FY2019-20 Planned |
             Future Cost | In Service Date
    project_total = Budget + FY2017-18 + FY2018-19 + FY2019-20 + Future Cost.
    Future Cost is captured but NOT stored as a year_YYYY column.
    """
    records = []
    current_dept = ''

    with pdfplumber.open(pdf_path) as pdf:
        for pg_num, pg in enumerate(pdf.pages, start=1):
            text = pg.extract_text() or ''
            if not is_project_page_a(text):
                continue

            dept = get_dept_from_text(text, 'Aprime')
            if dept:
                current_dept = dept

            tbl = pg.extract_table()
            if not tbl or len(tbl) < 2:
                continue

            # Find header row using strict first-cell check
            hdr_idx = find_hdr(tbl)
            if hdr_idx is None:
                continue

            headers = [str(c or '').replace('\n', ' ').strip() for c in tbl[hdr_idx]]

            col_map = {}
            year_cols = []
            future_cost_idx = None
            for i, h in enumerate(headers):
                hl = h.lower()
                if re.search(r'project', hl) and col_map.get('project_name') is None:
                    col_map['project_name'] = i
                elif re.search(r'service', hl):
                    col_map['project_type'] = i
                elif re.search(r'budget', hl) and 'previous_appropriations' not in col_map:
                    col_map['previous_appropriations'] = i
                elif re.search(r'future.*cost', hl):
                    future_cost_idx = i
                elif re.search(r'in.*service|completion', hl):
                    col_map['end_year'] = i
                fy = parse_fy_header(h)
                if fy:
                    year_cols.append((i, fy))

            if 'project_name' not in col_map:
                col_map['project_name'] = 0

            for row in tbl[hdr_idx + 1:]:
                if not any(row):
                    continue
                cells = [str(c or '').replace('\n', ' ').strip() for c in row]
                name = cells[col_map['project_name']]
                if not name or re.match(r'^(Total|Subtotal|Page|Fund)', name, re.I):
                    continue

                prev = clean_num(cells[col_map['previous_appropriations']]) if 'previous_appropriations' in col_map else ''
                yr_vals = {yr: clean_num(cells[ci]) for ci, yr in year_cols}
                future = clean_num(cells[future_cost_idx]) if future_cost_idx is not None else ''

                # Compute project_total = Budget + FY columns + Future Cost
                def to_f(v): return float(v) if v else 0.0
                total = to_f(prev) + sum(to_f(v) for v in yr_vals.values()) + to_f(future)
                project_total = str(int(total)) if total > 0 else ''

                if not project_total and not prev:
                    continue

                rec = {
                    'cip_year': cip_year,
                    'source_page': pg_num,
                    'department': current_dept,
                    'project_name': name,
                    'project_id': None,
                    'address_location': None,
                    'start_year': None,
                    'project_description': None,
                    'project_justification': None,
                    'project_type': cells[col_map['project_type']] if 'project_type' in col_map else None,
                    'previous_appropriations': prev,
                    'project_total': project_total,
                    'end_year': extract_end_year(cells[col_map['end_year']]) if 'end_year' in col_map else None,
                }
                rec.update(yr_vals)
                records.append(rec)

    return pd.DataFrame(records)

print('Format A\' parser defined.')

## Cell 6 — Format B parser (2018, landscape/rotated)

Pages are rotated 270°. Text characters are stored in reverse order in the PDF.
We use word-level extraction: reverse each word's characters, then map by `(x0, top)` coordinates.
- **x0** increases top→bottom across visual rows (each project = one x0 band)
- **top** locates the visual column (high=left, low=right in the landscape layout)

In [ ]:
# Format B column ranges: (top_min, top_max) in PDF coordinate space.
# Calibrated from page 201 of 2018.pdf (see notebook sampling output).
B_COL_RANGES = {
    'project_name':           (1100, 1400),
    'unit_number':            (1005, 1075),
    'fund_number':            (940,  1005),
    'council_district':       (875,  940),
    'capital_adopted':        (790,  875),   # previous_appropriations
    'expenditure_to_date':    (705,  790),
    'remaining_balance':      (620,  705),
    'year_2019':              (540,  620),   # FY2018-19
    'year_2020':              (460,  540),   # FY2019-20
    'year_2021':              (380,  460),   # FY2020-21
    'year_2022':              (295,  380),   # FY2021-22
    'year_2023':              (210,  295),   # FY2022-23
    'total_budget':           (100,  210),
}

B_YEAR_COLS = ['year_2019','year_2020','year_2021','year_2022','year_2023']

def classify_top(top: float) -> str:
    """Map a PDF 'top' coordinate to a column name for Format B."""
    for col, (lo, hi) in B_COL_RANGES.items():
        if lo <= top <= hi:
            return col
    return 'other'

def parse_format_b_page(pg, pg_num: int, cip_year: str, current_dept: list):
    """Parse a single rotated page from the 2018 PDF.
    Returns list of record dicts."""
    words = pg.extract_words(x_tolerance=5, y_tolerance=3)
    if not words:
        return []

    # Reverse each word's characters and bucket by column
    entries = []
    for w in words:
        x0  = w['x0']
        top = w['top']
        txt = w['text'][::-1]   # undo PDF character reversal
        col = classify_top(top)
        entries.append((x0, top, col, txt))

    # Header rows: x0 < 185
    # Check for department header (section title) at leftmost x0 band
    section_words = [(t, c) for x, t, c, txt in entries if x < 125]
    # Sort by top descending (left=high top in landscape)
    section_text = ' '.join(txt for x, t, c, txt in entries if x < 125)
    if re.search(r'IMPROVEMENT|DRAINAGE|FACILIT|WATER|LIBRARY|AVIATION', section_text, re.I):
        dept = re.sub(r'\s+CAPITAL\s+IMPROVEMENTS?\s*$', '', section_text, flags=re.I).strip().title()
        current_dept[0] = dept

    # Data rows: x0 >= 185
    data = [(x0, top, col, txt) for x0, top, col, txt in entries if x0 >= 185]

    # Cluster data by x0 (each project spans a narrow x0 range, ~28 units apart)
    # Use 15-unit tolerance: round x0 to nearest 15 to group multi-word cells
    from itertools import groupby
    def x0_key(item): return round(item[0] / 14) * 14
    data_sorted = sorted(data, key=lambda e: (x0_key(e), -e[1]))

    projects_raw = {}
    for key, group in groupby(data_sorted, key=lambda e: x0_key(e)):
        items = list(group)
        row_data = {}
        for _, top, col, txt in items:
            if col == 'other':
                continue
            row_data.setdefault(col, []).append(txt)
        if row_data:
            projects_raw[key] = row_data

    records = []
    for x0_cluster, row_data in sorted(projects_raw.items()):
        # Must have at least a project name or unit number to be a real project row
        name_words = row_data.get('project_name', [])
        unit_raw   = ' '.join(row_data.get('unit_number', []))

        # Unit number format: 'U_XXXX' → project_id = 'XXXX'
        uid = None
        for tok in unit_raw.split():
            m = re.match(r'U_([A-Z0-9]+)', tok)
            if m:
                uid = m.group(1)
                break

        if not name_words and not uid:
            continue

        # Project name: join words, sorted left-to-right in landscape (top descending)
        name = ' '.join(name_words).strip()
        # Skip pure header/section rows
        if re.match(r'^(Total|Fund|Unit|Project|Council|Capital)', name, re.I) and not uid:
            continue

        capital_adopted = clean_num(' '.join(row_data.get('capital_adopted', [])))
        total_budget    = clean_num(' '.join(row_data.get('total_budget', [])))

        if not name and not capital_adopted and not total_budget:
            continue

        rec = {
            'cip_year':               cip_year,
            'project_type':           None,
            'source_page':            pg_num,
            'department':             current_dept[0],
            'project_name':           name,
            'project_id':             uid,
            'address_location':       None,
            'start_year':             None,
            'end_year':               None,
            'project_description':    None,
            'project_justification':  None,
            'previous_appropriations': capital_adopted,
            'project_total':          total_budget,
        }
        for yr_col in B_YEAR_COLS:
            rec[yr_col] = clean_num(' '.join(row_data.get(yr_col, [])))
        records.append(rec)

    return records


def parse_format_b(pdf_path: Path, cip_year: str) -> pd.DataFrame:
    """Parse Format B (2018) PDF — landscape pages only."""
    records = []
    current_dept = ['']  # mutable container for cross-page state
    with pdfplumber.open(pdf_path) as pdf:
        for pg_num, pg in enumerate(pdf.pages, start=1):
            if pg.rotation != 270:
                continue
            records.extend(parse_format_b_page(pg, pg_num, cip_year, current_dept))
    return pd.DataFrame(records)

print('Format B parser defined.')

## Cell 7 — Format C parser (2019–2025)

In [ ]:
def parse_format_c(pdf_path: Path, cip_year: str) -> pd.DataFrame:
    """Parse Format C (2019-2025) PDFs.
    Table columns: Project | Service | Funding Source | Council District |
                   [Est. Comp./Completion] Date | Budget ITD / Budget as of [date] |
                   Spent or Committed | Remaining | FY YYYY-YY [Budget/Planned] |
                   [Future Costs] | Total Project Cost(s)
    project_id extracted from ' - XXXX' suffix in project name.
    """
    records = []
    current_dept = ''

    with pdfplumber.open(pdf_path) as pdf:
        for pg_num, pg in enumerate(pdf.pages, start=1):
            text = pg.extract_text() or ''
            if not is_project_page_c(text):
                continue

            dept = get_dept_from_text(text, 'C')
            if dept and not re.match(r'^(Project|Council|Service|FY)', dept, re.I):
                current_dept = dept

            tbl = pg.extract_table()
            if not tbl or len(tbl) < 2:
                continue

            # Find header row using strict first-cell check
            hdr_idx = find_hdr(tbl)
            if hdr_idx is None:
                continue

            headers = [str(c or '').replace('\n', ' ').strip() for c in tbl[hdr_idx]]

            col_map = {}
            year_cols = []
            for i, h in enumerate(headers):
                hl = h.lower()
                if re.match(r'project', hl) and 'project_name' not in col_map:
                    col_map['project_name'] = i
                elif re.search(r'service', hl):
                    col_map['project_type'] = i
                elif re.search(r'completion|est.*comp|comp.*date', hl):
                    col_map['end_year'] = i
                elif re.search(r'budget.*itd|itd.*budget|budget\s+as\s+of|capital.*adopt', hl):
                    col_map['previous_appropriations'] = i
                elif re.search(r'total.*project.*cost|total.*budget|total.*cost', hl):
                    col_map['project_total'] = i
                # Future Costs is NOT a year_YYYY column — skip it
                elif re.search(r'future.*cost', hl):
                    pass
                fy = parse_fy_header(h)
                if fy:
                    year_cols.append((i, fy))

            if 'project_name' not in col_map:
                col_map['project_name'] = 0

            for row in tbl[hdr_idx + 1:]:
                if not any(row):
                    continue
                cells = [str(c or '').replace('\n', ' ').strip() for c in row]
                raw_name = cells[col_map['project_name']]
                if not raw_name:
                    continue
                if re.match(r'^(Total|Subtotal|Fund|Use of)', raw_name, re.I):
                    continue

                # Extract project_id from name suffix
                clean_name, pid = extract_pid_from_name(raw_name)

                rec = {
                    'cip_year':              cip_year,
                    'source_page':           pg_num,
                    'department':            current_dept,
                    'project_name':          clean_name,
                    'project_id':            pid,
                    'address_location':      None,
                    'start_year':            None,
                    'project_description':   None,
                    'project_justification': None,
                    'project_type':          cells[col_map['project_type']] if 'project_type' in col_map else None,
                    'previous_appropriations': clean_num(cells[col_map['previous_appropriations']]) if 'previous_appropriations' in col_map else None,
                    'project_total':         clean_num(cells[col_map['project_total']]) if 'project_total' in col_map else None,
                    'end_year':              extract_end_year(cells[col_map['end_year']]) if 'end_year' in col_map else None,
                }
                for ci, yr in year_cols:
                    rec[yr] = clean_num(cells[ci])
                records.append(rec)

    return pd.DataFrame(records)

print('Format C parser defined.')

## Cell 8 — Main processing loop

In [ ]:
PARSERS = {
    'A':      parse_format_a,
    'Aprime': parse_format_aprime,
    'B':      parse_format_b,
    'C':      parse_format_c,
}

summary = []

for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    stem = pdf_path.stem
    fmt  = detect_format(stem)
    print(f'Processing {stem}.pdf  (Format {fmt}) ...', end=' ')

    df = PARSERS[fmt](pdf_path, stem)

    if df.empty:
        print(f'WARNING: no rows extracted!')
        summary.append({'pdf': stem, 'format': fmt, 'rows': 0})
        continue

    # Ensure all BASE_COLS present; order columns: base cols first, then year_YYYY sorted
    year_col_names = sorted([c for c in df.columns if c.startswith('year_')])
    final_cols = BASE_COLS + year_col_names
    for c in final_cols:
        if c not in df.columns:
            df[c] = None
    df = df[final_cols]

    out_path = CSV_DIR / f'{stem}.csv'
    df.to_csv(out_path, index=False)
    print(f'{len(df):,} rows → {out_path.name}')
    summary.append({'pdf': stem, 'format': fmt, 'rows': len(df), 'year_cols': year_col_names})

print('\nDone.')

## Cell 9 — Output summary and null-rate diagnostics

In [ ]:
print(f'{"PDF":<10} {"Format":<8} {"Rows":>6}  Year columns')
print('-' * 70)
for s in summary:
    yc = ', '.join(s.get('year_cols', []))
    print(f"{s['pdf']:<10} {s['format']:<8} {s['rows']:>6}  {yc}")

print('\n--- Null rates for key columns (>30%) ---')
for s in summary:
    csv_path = CSV_DIR / f"{s['pdf']}.csv"
    if not csv_path.exists() or s['rows'] == 0:
        continue
    df = pd.read_csv(csv_path, dtype=str)
    n = len(df)
    check_cols = ['project_name','project_id','previous_appropriations','project_total','department']
    nulls = {c: f"{df[c].isna().sum()/n:.0%}" for c in check_cols if c in df.columns and df[c].isna().sum()/n > 0.3}
    if nulls:
        print(f"  {s['pdf']}: {nulls}")